In [2]:
# Catastro API: fetch municipios and vías (street names) for code → name mapping
# API key from agent/.env (CATASTRO_API_KEY)
# Docs: https://catastro-api.es/docs

import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

API_KEY = 'ctr_f6b286671d2fc527cc904ba4b53c5185dde1555c86b8a983e536464af4785103'

BASE_URL = "https://api.catastro-api.es"  # adjust if docs specify another base

HEADERS = {"X-API-Key": API_KEY}

In [3]:
# 1. Fetch all municipios
resp = requests.get(f"{BASE_URL}/api/callejero/municipios", headers=HEADERS, params={"provincia": "VALENCIA"}, timeout=30)
resp.raise_for_status()
municipios = resp.json()
# Inspect structure (might be a list or wrapped in a key)
if isinstance(municipios, list):
    print(f"Got {len(municipios)} municipios")
    if municipios:
        print("Sample:", municipios[0])
else:
    print("Keys:", municipios.keys() if isinstance(municipios, dict) else type(municipios))
    print(municipios)

Keys: dict_keys(['numeroMunicipios', 'municipios', 'errores'])
{'numeroMunicipios': 266, 'municipios': [{'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '001', 'nombreMunicipio': 'ADEMUZ'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '002', 'nombreMunicipio': 'ADOR'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '004', 'nombreMunicipio': 'AGULLENT'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '042', 'nombreMunicipio': 'AIELO DE MALFERIT'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '043', 'nombreMunicipio': 'AIELO DE RUGAT'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '005', 'nombreMunicipio': 'ALAQUAS'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '006', 'nombreMunicipio': 'ALBAIDA'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '007', 'nombreMunicipio': 'ALBAL'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '008', 'nombreMunicipio': 'ALBALAT DE LA RIBERA'}, {'codigoProvinciaIne': '46', 'codigoMunicipioMeh': '009', 'nombreMunic

In [4]:
# 2. Normalize municipios list (use nombreMunicipio for vías request)
if isinstance(municipios, dict):
    municipios_list = municipios.get("municipios", [])
else:
    municipios_list = municipios

if not municipios_list:
    raise ValueError("No municipios returned from API")

# Vías request uses: provincia="VALENCIA", municipio=nombreMunicipio
PROVINCIA = "VALENCIA"
print(f"Will request vías with provincia={PROVINCIA} and municipio=<nombreMunicipio> for {len(municipios_list)} municipios")

Will request vías with provincia=VALENCIA and municipio=<nombreMunicipio> for 266 municipios


In [5]:
vias_rows = []
for i, m in enumerate(municipios_list):
    if not isinstance(m, dict):
        continue
    nombre_municipio = m.get("nombreMunicipio")
    if not nombre_municipio:
        continue
    try:
        r = requests.get(
            f"{BASE_URL}/api/callejero/vias",
            headers=HEADERS,
            params={"provincia": PROVINCIA, "municipio": nombre_municipio},
            timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        # Normalize to list of vias (response might be {"vias": [...]} or direct list)
        vias = data if isinstance(data, list) else data.get("vias", data.get("data", []))
        for v in vias:
            if isinstance(v, dict):
                vias_rows.append({
                    "provincia": PROVINCIA,
                    "nombreMunicipio": nombre_municipio,
                    "CodigoVia": v.get("CodigoVia") or v.get("codigoVia"),
                    "tipoVia": v.get("tipoVia") or v.get("TipoVia"),
                    "nombreVia": v.get("nombreVia") or v.get("NombreVia"),
                })
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1} municipios, {len(vias_rows)} vías so far")
    except Exception as e:
        print(f"Error municipio {nombre_municipio}: {e}")
    time.sleep(2)  # rate limit

print(f"Total vías: {len(vias_rows)}")

Processed 50 municipios, 8396 vías so far
Processed 100 municipios, 17107 vías so far
Processed 150 municipios, 23679 vías so far
Processed 200 municipios, 31620 vías so far
Processed 250 municipios, 39966 vías so far
Total vías: 45164


In [7]:
# 3. Build code → street name mapping (and save for agent use)
import pandas as pd

df_vias = pd.DataFrame(vias_rows)
# via_code for matching with catastro parcels: often "provincia + municipio + CodigoVia" or just CodigoVia
df_vias["via_code"] = df_vias["CodigoVia"].astype(str)
df_vias["street_name"] = (df_vias["tipoVia"].fillna("").astype(str) + " " + df_vias["nombreVia"].fillna("").astype(str)).str.strip()
df_vias.head(10)

,provincia,nombreMunicipio,CodigoVia,tipoVia,nombreVia,via_code,street_name
0,VALENCIA,ADEMUZ,91,DS,ADEMUZ-OT,91,DS ADEMUZ-OT
1,VALENCIA,ADEMUZ,90,DS,ADEMUZ-RS,90,DS ADEMUZ-RS
2,VALENCIA,ADEMUZ,1,CL,BLANCO,1,CL BLANCO
3,VALENCIA,ADEMUZ,104,CL,BOHILGUES,104,CL BOHILGUES
4,VALENCIA,ADEMUZ,2,CL,BOTICARIO,2,CL BOTICARIO
5,VALENCIA,ADEMUZ,3,CL,CALDERAS,3,CL CALDERAS
6,VALENCIA,ADEMUZ,23,CM,CERRADA,23,CM CERRADA
7,VALENCIA,ADEMUZ,4,CL,CRUCES,4,CL CRUCES
8,VALENCIA,ADEMUZ,36,DS,DISEMINADOS,36,DS DISEMINADOS
9,VALENCIA,ADEMUZ,31,CL,DOMICILIO PROPIO,31,CL DOMICILIO PROPIO


In [8]:
# Save for agent: streets table can be populated from this
out_path = Path.cwd() / "data" / "catastro_vias.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_vias.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {len(df_vias)} vías to {out_path}")

Saved 45164 vías to /Users/miguelfa/Desktop/rooster-capstone-project/pipeline/catastro/data/catastro_vias.csv
